In [25]:
# loader scripts

import requests
import pandas as pd

def to_tz(df_, time_col, tz_offset, tz_name):
    return (df_
    .groupby(tz_offset)
    [time_col]
    .transform(lambda s: pd.to_datetime(s)
    .dt.tz_localize(s.name, ambiguous=True)
    .dt.tz_convert(tz_name))
    )



def load_fuel_economy_data() -> pd.DataFrame :
    url = 'https://www.fueleconomy.gov/feg/epadata/vehicles.csv'
    cols = ['year', 'make', 'model', 'trany', 'drive', 'VClass', 'eng_dscr',
            'barrels08', 'city08', 'comb08', 'range', 'evMotor', 'cylinders', 'displ', 'fuelCost08',
            'fuelType', 'highway08', 'trans_dscr','createdOn']

    raw =  pd.read_csv(url)
    autos = (raw
        .loc[:, cols]
        .assign(
            # Extract Timezone Offset (e.g., EDT -> EST5EDT)
            offset=(raw.createdOn.str.extract(r'\d\d:\d\d (?P<offset>[A-Z]{3}?)')
                    ['offset'].replace('EDT', 'EST5EDT')),
            
            # Reconstruct date string (removing the original TZ name for parsing)
            str_date=(raw.createdOn.str.slice(4, 19) + " " +
                    raw.createdOn.str.slice(-4)),
            
            # Convert to localized datetime
            createdOn=lambda df_: to_tz(df_, 'str_date', 'offset', 'America/New_York')
        )
        .drop(columns=['offset', 'str_date']) # Clean up temp columns
        )
    return autos

In [26]:
# analyzing the data
df = load_fuel_economy_data()

df.head()

C:\Users\dsgou\AppData\Local\Temp\ipykernel_34780\2720021934.py:23: DtypeWarning: Columns (69,71,72,73,74,75,77,80) have mixed types. Specify dtype option on import or set low_memory=False.
  raw =  pd.read_csv(url)


,year,make,model,trany,drive,VClass,eng_dscr,barrels08,city08,comb08,range,evMotor,cylinders,displ,fuelCost08,fuelType,highway08,trans_dscr,createdOn
0,1985,Alfa Romeo,Spider Veloce 2000,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(FFS),14.167143,19,21,0,NaN,4.0,2.0,2850,Regular,25,NaN,2013-01-01 00:00:00-05:00
1,1985,Ferrari,Testarossa,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(GUZZLER),27.046364,9,11,0,NaN,12.0,4.9,5450,Regular,14,NaN,2013-01-01 00:00:00-05:00
2,1985,Dodge,Charger,Manual 5-spd,Front-Wheel Drive,Subcompact Cars,(FFS),11.018889,23,27,0,NaN,4.0,2.2,2200,Regular,33,SIL,2013-01-01 00:00:00-05:00
3,1985,Dodge,B150/B250 Wagon 2WD,Automatic 3-spd,Rear-Wheel Drive,Vans,NaN,27.046364,10,11,0,NaN,8.0,5.2,5450,Regular,12,NaN,2013-01-01 00:00:00-05:00
4,1993,Subaru,Legacy AWD Turbo,Manual 5-spd,4-Wheel or All-Wheel Drive,Compact Cars,"(FFS,TRBO)",15.658421,17,19,0,NaN,4.0,2.2,3650,Premium,23,NaN,2013-01-01 00:00:00-05:00


In [27]:
df.isna().mean()*100

year           0.000000
make           0.000000
model          0.000000
trany          0.022068
drive          2.379328
VClass         0.000000
eng_dscr      36.530514
barrels08      0.000000
city08         0.000000
comb08         0.000000
range          0.000000
evMotor       93.084701
cylinders      2.951089
displ          2.947077
fuelCost08     0.000000
fuelType       0.000000
highway08      0.000000
trans_dscr    69.819043
createdOn      0.000000
dtype: float64

In [28]:
# find all vehicles that have missing cylinders info
df[df['cylinders'].isna()]

# This is likely going to be EVs where the data is expected to be missing
# To "clean" our data we will want to set this value to something meaningful
# Usually we consult SMEs to understand what would be the neutral state of the data
# when values are missing
# In this case 0 is the right answer but it could easily be mean or median depending 
# on the semantic nature of the data

,year,make,model,trany,drive,VClass,eng_dscr,barrels08,city08,comb08,range,evMotor,cylinders,displ,fuelCost08,fuelType,highway08,trans_dscr,createdOn
7138,2000,Nissan,Altra EV,NaN,NaN,Midsize Station Wagons,NaN,0.0960,81,85,90,62 KW AC Induction,NaN,NaN,900,Electricity,91,NaN,2013-01-01 00:00:00-05:00
7139,2000,Toyota,RAV4 EV,NaN,2-Wheel Drive,Sport Utility Vehicle - 2WD,NaN,0.1128,81,72,88,50 KW DC,NaN,NaN,1050,Electricity,64,NaN,2013-01-01 00:00:00-05:00
8143,2001,Toyota,RAV4 EV,NaN,2-Wheel Drive,Sport Utility Vehicle - 2WD,NaN,0.1128,81,72,88,50 KW DC,NaN,NaN,1050,Electricity,64,NaN,2013-01-01 00:00:00-05:00
8144,2001,Ford,Th!nk,NaN,NaN,Two Seaters,NaN,0.1248,74,65,29,27 KW AC Induction,NaN,NaN,1150,Electricity,58,NaN,2013-01-01 00:00:00-05:00
8146,2001,Ford,Explorer USPS Electric,NaN,2-Wheel Drive,Sport Utility Vehicle - 2WD,NaN,0.2088,45,39,38,67 KW AC Induction,NaN,NaN,1950,Electricity,33,NaN,2013-01-01 00:00:00-05:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44321,2026,Toyota,bZ Woodland AWD 235/60R18,Automatic (A1),All-Wheel Drive,Small Sport Utility Vehicle 4WD,NaN,0.0696,127,117,281,167 kW AC Synchronous,NaN,NaN,650,Electricity,107,NaN,2026-03-18 00:00:00-04:00
44322,2026,Toyota,bZ Woodland AWD 235/65R18,Automatic (A1),All-Wheel Drive,Small Sport Utility Vehicle 4WD,NaN,0.0744,117,109,260,167 kW AC Synchronous,NaN,NaN,700,Electricity,100,NaN,2026-03-18 00:00:00-04:00
44323,2026,Toyota,C-HR AWD 18inch,Automatic (A1),All-Wheel Drive,Small Sport Utility Vehicle 4WD,NaN,0.0696,127,117,287,87 and 167 kW AC Synchronous,NaN,NaN,650,Electricity,107,NaN,2026-03-18 00:00:00-04:00
44324,2026,Toyota,C-HR AWD 20inch,Automatic (A1),All-Wheel Drive,Small Sport Utility Vehicle 4WD,NaN,0.0720,122,112,273,87 and 167 kW AC Synchronous,NaN,NaN,650,Electricity,102,NaN,2026-03-18 00:00:00-04:00


In [29]:
# assign the missing values to 0
df.assign(cylinders=df.cylinders.fillna(0))

,year,make,model,trany,drive,VClass,eng_dscr,barrels08,city08,comb08,range,evMotor,cylinders,displ,fuelCost08,fuelType,highway08,trans_dscr,createdOn
0,1985,Alfa Romeo,Spider Veloce 2000,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(FFS),14.167143,19,21,0,NaN,4.0,2.0,2850,Regular,25,NaN,2013-01-01 00:00:00-05:00
1,1985,Ferrari,Testarossa,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(GUZZLER),27.046364,9,11,0,NaN,12.0,4.9,5450,Regular,14,NaN,2013-01-01 00:00:00-05:00
2,1985,Dodge,Charger,Manual 5-spd,Front-Wheel Drive,Subcompact Cars,(FFS),11.018889,23,27,0,NaN,4.0,2.2,2200,Regular,33,SIL,2013-01-01 00:00:00-05:00
3,1985,Dodge,B150/B250 Wagon 2WD,Automatic 3-spd,Rear-Wheel Drive,Vans,NaN,27.046364,10,11,0,NaN,8.0,5.2,5450,Regular,12,NaN,2013-01-01 00:00:00-05:00
4,1993,Subaru,Legacy AWD Turbo,Manual 5-spd,4-Wheel or All-Wheel Drive,Compact Cars,"(FFS,TRBO)",15.658421,17,19,0,NaN,4.0,2.2,3650,Premium,23,NaN,2013-01-01 00:00:00-05:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49841,1993,Subaru,Legacy,Automatic 4-spd,Front-Wheel Drive,Compact Cars,(FFS),13.523182,19,22,0,NaN,4.0,2.2,2700,Regular,26,CLKUP,2013-01-01 00:00:00-05:00
49842,1993,Subaru,Legacy,Manual 5-spd,Front-Wheel Drive,Compact Cars,(FFS),12.935217,20,23,0,NaN,4.0,2.2,2600,Regular,28,NaN,2013-01-01 00:00:00-05:00
49843,1993,Subaru,Legacy AWD,Automatic 4-spd,4-Wheel or All-Wheel Drive,Compact Cars,(FFS),14.167143,18,21,0,NaN,4.0,2.2,2850,Regular,24,CLKUP,2013-01-01 00:00:00-05:00
49844,1993,Subaru,Legacy AWD,Manual 5-spd,4-Wheel or All-Wheel Drive,Compact Cars,(FFS),14.167143,18,21,0,NaN,4.0,2.2,2850,Regular,24,NaN,2013-01-01 00:00:00-05:00


In [30]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn import set_config

# set output to pandas 
set_config(transform_output='pandas')

# create a pipeline for cylinders
cyl_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value=0)) # assign the default value to 0 for missing values
])

cyl_pipe.fit_transform(df[['cylinders']])


,cylinders
0,4.0
1,12.0
2,4.0
3,8.0
4,4.0
...,...
49841,4.0
49842,4.0
49843,4.0
49844,4.0


In [31]:
# see where it "filled" the missing values

(cyl_pipe
.fit_transform(df[['cylinders']]) # the transformed values
.loc[df.cylinders.isna()] # from the original dataframes all the na columns for cylinders
)

,cylinders
7138,0.0
7139,0.0
8143,0.0
8144,0.0
8146,0.0
...,...
44321,0.0
44322,0.0
44323,0.0
44324,0.0


In [32]:
# The prod like pipeline if we really decided to use cylinder and displacement columns

from sklearn.compose import ColumnTransformer

# create the imputers
cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0) # fill missing values with 0
displ_imputer = SimpleImputer(strategy='median') # fill missing values with median


# create the preprocessors
preprocessor = ColumnTransformer(
    transformers=[
        ('cyl_imputer', cylinders_imputer, ['cylinders']), # creates a new columnn with the transform applied
        ('displ_imputer', displ_imputer, ['displ'])
    ],
    remainder='passthrough' # passes through all the existing columns
)

# create the pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# fit and transform the data
pipeline.fit_transform(df)


,cyl_imputer__cylinders,displ_imputer__displ,remainder__year,remainder__make,remainder__model,remainder__trany,remainder__drive,remainder__VClass,remainder__eng_dscr,remainder__barrels08,remainder__city08,remainder__comb08,remainder__range,remainder__evMotor,remainder__fuelCost08,remainder__fuelType,remainder__highway08,remainder__trans_dscr,remainder__createdOn
0,4.0,2.0,1985,Alfa Romeo,Spider Veloce 2000,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(FFS),14.167143,19,21,0,NaN,2850,Regular,25,NaN,2013-01-01 00:00:00-05:00
1,12.0,4.9,1985,Ferrari,Testarossa,Manual 5-spd,Rear-Wheel Drive,Two Seaters,(GUZZLER),27.046364,9,11,0,NaN,5450,Regular,14,NaN,2013-01-01 00:00:00-05:00
2,4.0,2.2,1985,Dodge,Charger,Manual 5-spd,Front-Wheel Drive,Subcompact Cars,(FFS),11.018889,23,27,0,NaN,2200,Regular,33,SIL,2013-01-01 00:00:00-05:00
3,8.0,5.2,1985,Dodge,B150/B250 Wagon 2WD,Automatic 3-spd,Rear-Wheel Drive,Vans,NaN,27.046364,10,11,0,NaN,5450,Regular,12,NaN,2013-01-01 00:00:00-05:00
4,4.0,2.2,1993,Subaru,Legacy AWD Turbo,Manual 5-spd,4-Wheel or All-Wheel Drive,Compact Cars,"(FFS,TRBO)",15.658421,17,19,0,NaN,3650,Premium,23,NaN,2013-01-01 00:00:00-05:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49841,4.0,2.2,1993,Subaru,Legacy,Automatic 4-spd,Front-Wheel Drive,Compact Cars,(FFS),13.523182,19,22,0,NaN,2700,Regular,26,CLKUP,2013-01-01 00:00:00-05:00
49842,4.0,2.2,1993,Subaru,Legacy,Manual 5-spd,Front-Wheel Drive,Compact Cars,(FFS),12.935217,20,23,0,NaN,2600,Regular,28,NaN,2013-01-01 00:00:00-05:00
49843,4.0,2.2,1993,Subaru,Legacy AWD,Automatic 4-spd,4-Wheel or All-Wheel Drive,Compact Cars,(FFS),14.167143,18,21,0,NaN,2850,Regular,24,CLKUP,2013-01-01 00:00:00-05:00
49844,4.0,2.2,1993,Subaru,Legacy AWD,Manual 5-spd,4-Wheel or All-Wheel Drive,Compact Cars,(FFS),14.167143,18,21,0,NaN,2850,Regular,24,NaN,2013-01-01 00:00:00-05:00
